In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/trabalhodothompson/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/trabalhodothompson/crime_incidents.csv,crime_incidents.csv,7524157,1793728875000
dbfs:/Volumes/workspace/default/trabalhodothompson/forecast_sequences.csv,forecast_sequences.csv,10409970,1793728875000
dbfs:/Volumes/workspace/default/trabalhodothompson/locations.csv,locations.csv,4871,1793728873000
dbfs:/Volumes/workspace/default/trabalhodothompson/sightings.csv,sightings.csv,21469643,1793728877000
dbfs:/Volumes/workspace/default/trabalhodothompson/weather.csv,weather.csv,773911,1793728873000
dbfs:/Volumes/workspace/default/trabalhodothompson/witnesses_public.csv,witnesses_public.csv,1223915,1793728873000


In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/trabalhodothompson/crime_incidents.csv",
    header=True,
    inferSchema=True,
)
df.printSchema()
print(df.count())
display(df.limit(10))

root
 |-- crime_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- time_of_day: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- borough: string (nullable = true)
 |-- district: string (nullable = true)
 |-- h3_cell: string (nullable = true)
 |-- crime_type: string (nullable = true)
 |-- severity_score: double (nullable = true)
 |-- victim_count: integer (nullable = true)
 |-- weapon_reported: boolean (nullable = true)
 |-- police_called: boolean (nullable = true)
 |-- estimated_police_eta_min: double (nullable = true)
 |-- public_visibility_score: double (nullable = true)
 |-- is_decoy_event: boolean (nullable = true)

46000


crime_id,timestamp,date,hour,time_of_day,latitude,longitude,borough,district,h3_cell,crime_type,severity_score,victim_count,weapon_reported,police_called,estimated_police_eta_min,public_visibility_score,is_decoy_event
CRM_00000000,2025-01-01T00:02:48.000Z,2025-01-01,0,night,40.67339,-73.9656,Brooklyn,Brooklyn_Central_9,882a100db5fffff,vehicle_theft,3.92,0,false,true,5.5,0.475,false
CRM_00000001,2025-01-01T00:12:33.000Z,2025-01-01,0,night,40.84752,-73.8472,Bronx,Bronx_Southeast_4,882a1001e9fffff,suspicious_activity,2.17,0,false,true,6.3,0.625,true
CRM_00000002,2025-01-01T00:38:05.000Z,2025-01-01,0,night,40.85656,-73.92903,Manhattan,Manhattan_Northwest_8,882a100ae1fffff,burglary,4.97,1,false,true,4.8,0.311,false
CRM_00000003,2025-01-01T00:49:44.000Z,2025-01-01,0,night,40.7238,-74.00853,Manhattan,Manhattan_Central_0,882a1072c5fffff,pursuit,6.66,0,true,true,4.3,0.723,false
CRM_00000004,2025-01-01T00:51:36.000Z,2025-01-01,0,night,40.81054,-73.95247,Manhattan,Manhattan_South_5,882a1008cbfffff,major_incident,10.0,7,true,true,3.3,0.553,false
CRM_00000005,2025-01-01T00:59:17.000Z,2025-01-01,0,night,40.61377,-73.83301,Queens,Queens_Northeast_2,882a10390bfffff,vehicle_theft,4.86,0,false,true,10.5,0.619,false
CRM_00000006,2025-01-01T01:00:37.000Z,2025-01-01,1,night,40.85087,-73.93494,Manhattan,Manhattan_Northwest_8,882a100ae7fffff,burglary,5.44,0,false,true,5.8,0.198,false
CRM_00000007,2025-01-01T01:03:33.000Z,2025-01-01,1,night,40.84944,-73.93873,Manhattan,Manhattan_Northwest_8,882a100ae7fffff,property_damage,3.34,0,false,true,4.7,0.468,false
CRM_00000008,2025-01-01T01:12:34.000Z,2025-01-01,1,night,40.69539,-73.76993,Queens,Queens_West_7,882a1005a5fffff,suspicious_activity,3.26,0,false,true,7.6,0.384,false
CRM_00000009,2025-01-01T01:14:19.000Z,2025-01-01,1,night,40.67705,-73.97227,Brooklyn,Brooklyn_Central_9,882a100da3fffff,vehicle_theft,4.32,0,false,true,5.3,0.465,false


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
base = "/Volumes/workspace/default/trabalhodothompson/"
arquivos = ["crime_incidents", "sightings", "witnesses_public", "locations", "weather", "forecast_sequences"]

for nome in arquivos:
    d = spark.read.csv(f"{base}{nome}.csv", header=True, inferSchema=True)
    print(f"===== {nome} | {d.count()} linhas | {len(d.columns)} colunas =====")
    d.printSchema()

===== crime_incidents | 46000 linhas | 18 colunas =====
root
 |-- crime_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- time_of_day: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- borough: string (nullable = true)
 |-- district: string (nullable = true)
 |-- h3_cell: string (nullable = true)
 |-- crime_type: string (nullable = true)
 |-- severity_score: double (nullable = true)
 |-- victim_count: integer (nullable = true)
 |-- weapon_reported: boolean (nullable = true)
 |-- police_called: boolean (nullable = true)
 |-- estimated_police_eta_min: double (nullable = true)
 |-- public_visibility_score: double (nullable = true)
 |-- is_decoy_event: boolean (nullable = true)

===== sightings | 86631 linhas | 33 colunas =====
root
 |-- sighting_id: string (nullable = true)
 |-- split: string (nullable = true)
 |-- timestamp: times

In [0]:
from pyspark.sql import functions as F

base = "/Volumes/workspace/default/trabalhodothompson/"
arquivos = ["crime_incidents", "sightings", "witnesses_public", "locations", "weather", "forecast_sequences"]

for nome in arquivos:
    d = spark.read.csv(f"{base}{nome}.csv", header=True, inferSchema=True)

    # Spark/Delta não aceita caracteres como "-" em nomes de coluna (ex.: hist_t-5_lat)
    for c in d.columns:
        d = d.withColumnRenamed(c, c.replace("-", "_").replace(" ", "_"))

    d = (d.withColumn("_ingestao_em", F.current_timestamp())
          .withColumn("_arquivo_origem", F.lit(f"{nome}.csv")))

    d.write.mode("overwrite").saveAsTable(f"workspace.bronze.{nome}")
    print(f"bronze.{nome} criada: {d.count()} linhas")

bronze.crime_incidents criada: 46000 linhas
bronze.sightings criada: 86631 linhas
bronze.witnesses_public criada: 27000 linhas
bronze.locations criada: 46 linhas
bronze.weather criada: 13104 linhas
bronze.forecast_sequences criada: 54606 linhas


In [0]:
chaves = {
    "crime_incidents": "crime_id",
    "sightings": "sighting_id",
    "witnesses_public": "witness_id",
    "locations": "zone_id",
    "weather": "weather_id",
    "forecast_sequences": "sequence_id",
}

for nome, chave in chaves.items():
    t = spark.table(f"workspace.bronze.{nome}")
    total = t.count()
    unicos = t.select(chave).distinct().count()
    print(f"{nome}: {total} linhas | {unicos} chaves únicas | duplicadas: {total - unicos}")

crime_incidents: 46000 linhas | 46000 chaves únicas | duplicadas: 0
sightings: 86631 linhas | 86631 chaves únicas | duplicadas: 0
witnesses_public: 27000 linhas | 27000 chaves únicas | duplicadas: 0
locations: 46 linhas | 46 chaves únicas | duplicadas: 0
weather: 13104 linhas | 13104 chaves únicas | duplicadas: 0
forecast_sequences: 54606 linhas | 54606 chaves únicas | duplicadas: 0


In [0]:
from pyspark.sql import functions as F

for nome in ["crime_incidents", "sightings", "witnesses_public", "locations", "weather"]:
    t = spark.table(f"workspace.bronze.{nome}")
    total = t.count()
    cols = [c for c in t.columns if not c.startswith("_")]
    nulos = t.select([
        F.round(F.sum(F.col(f"`{c}`").isNull().cast("int")) / total * 100, 2).alias(c)
        for c in cols
    ])
    print(f"===== % de nulos em {nome} =====")
    display(nulos)

===== % de nulos em crime_incidents =====


crime_id,timestamp,date,hour,time_of_day,latitude,longitude,borough,district,h3_cell,crime_type,severity_score,victim_count,weapon_reported,police_called,estimated_police_eta_min,public_visibility_score,is_decoy_event
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


===== % de nulos em sightings =====


sighting_id,split,timestamp,latitude,longitude,borough,district,h3_cell,report_type,witness_count,unique_source_count,tracker_confidence,photo_evidence,mean_photo_quality,video_evidence,audio_evidence,web_residue_found,police_report_linked,crime_nearby,nearest_crime_id,distance_to_nearest_crime_m,minutes_from_nearest_crime,estimated_speed_kmh,estimated_altitude_m,movement_direction,weather_condition,visibility_m,crowd_density_score,report_source,location_spread_m,median_report_delay_s,verification_status,is_verified
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.99,0.0,0.0,0.0,0.0,0.0,92.99,92.99,92.99,17.49,46.26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


===== % de nulos em witnesses_public =====


witness_id,home_borough,total_reports,reports_with_photo,avg_statement_certainty,preferred_report_channel
0.0,0.0,0.0,0.0,0.02,0.0


===== % de nulos em locations =====


zone_id,borough,district,h3_cell,centroid_lat,centroid_lon,population_density_score,nightlife_score,commercial_score,residential_score,patrol_activity_score
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


===== % de nulos em weather =====


weather_id,date,hour,temperature_c,precipitation_mm,wind_kmh,visibility_m,cloud_cover,condition
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [0]:
from pyspark.sql import functions as F

for nome in ["crime_incidents", "sightings", "witnesses_public", "locations", "weather"]:
    t = spark.table(f"workspace.bronze.{nome}")
    total = t.count()
    cols = [c for c in t.columns if not c.startswith("_")]
    contagem = t.select([F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in cols]).first().asDict()
    com_nulos = {c: f"{v / total * 100:.2f}%" for c, v in contagem.items() if v and v > 0}
    print(f"{nome}: {com_nulos if com_nulos else 'nenhuma coluna com nulos'}")

crime_incidents: nenhuma coluna com nulos
sightings: {'mean_photo_quality': '31.99%', 'nearest_crime_id': '92.99%', 'distance_to_nearest_crime_m': '92.99%', 'minutes_from_nearest_crime': '92.99%', 'estimated_speed_kmh': '17.49%', 'estimated_altitude_m': '46.26%'}
witnesses_public: {'avg_statement_certainty': '0.02%'}
locations: nenhuma coluna com nulos
weather: nenhuma coluna com nulos


In [0]:
from pyspark.sql import functions as F

for nome in ["crime_incidents", "sightings", "witnesses_public", "locations", "weather"]:
    t = spark.table(f"workspace.bronze.{nome}")
    total = t.count()
    cols = [c for c in t.columns if not c.startswith("_")]
    contagem = t.select([F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in cols]).first().asDict()
    com_nulos = {c: f"{v / total * 100:.2f}%" for c, v in contagem.items() if v and v > 0}
    print(f"{nome}: {com_nulos if com_nulos else 'nenhuma coluna com nulos'}")

    display(spark.table("workspace.bronze.sightings").select(
    "latitude", "longitude", "tracker_confidence", "estimated_speed_kmh",
    "estimated_altitude_m", "visibility_m", "distance_to_nearest_crime_m",
    "minutes_from_nearest_crime", "witness_count"
).summary("min", "25%", "50%", "75%", "max"))

display(spark.table("workspace.bronze.crime_incidents").select(
    "latitude", "longitude", "hour", "severity_score", "victim_count",
    "estimated_police_eta_min", "public_visibility_score"
).summary("min", "25%", "50%", "75%", "max"))

display(spark.table("workspace.bronze.weather").select(
    "temperature_c", "precipitation_mm", "wind_kmh", "visibility_m", "cloud_cover"
).summary("min", "25%", "50%", "75%", "max"))

crime_incidents: nenhuma coluna com nulos


summary,latitude,longitude,tracker_confidence,estimated_speed_kmh,estimated_altitude_m,visibility_m,distance_to_nearest_crime_m,minutes_from_nearest_crime,witness_count
min,40.49166,-74.26064,0.208,0.0,0.0,154.0,0.0,-89.47,1
25%,40.63235,-73.97268,0.448,16.5,24.00909090909091,6959.0,0.0,3.82,1
50%,40.71906,-73.92991,0.518,32.2,43.2,8615.0,0.0,5.6,3
75%,40.81528,-73.87309,0.592,54.82000000000001,66.6,10319.0,0.0,8.18,5
max,40.91892,-73.70623,0.869,908.3499999999999,519.6,11998.0,1752.6,89.63,49


sightings: {'mean_photo_quality': '31.99%', 'nearest_crime_id': '92.99%', 'distance_to_nearest_crime_m': '92.99%', 'minutes_from_nearest_crime': '92.99%', 'estimated_speed_kmh': '17.49%', 'estimated_altitude_m': '46.26%'}


summary,latitude,longitude,tracker_confidence,estimated_speed_kmh,estimated_altitude_m,visibility_m,distance_to_nearest_crime_m,minutes_from_nearest_crime,witness_count
min,40.49166,-74.26064,0.208,0.0,0.0,154.0,0.0,-89.47,1
25%,40.63235,-73.97268,0.448,16.5,24.00909090909091,6959.0,0.0,3.82,1
50%,40.71906,-73.92991,0.518,32.2,43.2,8615.0,0.0,5.6,3
75%,40.81528,-73.87309,0.592,54.82000000000001,66.6,10319.0,0.0,8.18,5
max,40.91892,-73.70623,0.869,908.3499999999999,519.6,11998.0,1752.6,89.63,49


witnesses_public: {'avg_statement_certainty': '0.02%'}


summary,latitude,longitude,tracker_confidence,estimated_speed_kmh,estimated_altitude_m,visibility_m,distance_to_nearest_crime_m,minutes_from_nearest_crime,witness_count
min,40.49166,-74.26064,0.208,0.0,0.0,154.0,0.0,-89.47,1
25%,40.63235,-73.97268,0.448,16.5,24.00909090909091,6959.0,0.0,3.82,1
50%,40.71906,-73.92991,0.518,32.2,43.2,8615.0,0.0,5.6,3
75%,40.81528,-73.87309,0.592,54.82000000000001,66.6,10319.0,0.0,8.18,5
max,40.91892,-73.70623,0.869,908.3499999999999,519.6,11998.0,1752.6,89.63,49


... (2 outputs left out)

weather: nenhuma coluna com nulos


summary,latitude,longitude,tracker_confidence,estimated_speed_kmh,estimated_altitude_m,visibility_m,distance_to_nearest_crime_m,minutes_from_nearest_crime,witness_count
min,40.49166,-74.26064,0.208,0.0,0.0,154.0,0.0,-89.47,1
25%,40.63235,-73.97268,0.448,16.5,24.00909090909091,6959.0,0.0,3.82,1
50%,40.71906,-73.92991,0.518,32.2,43.2,8615.0,0.0,5.6,3
75%,40.81528,-73.87309,0.592,54.82000000000001,66.6,10319.0,0.0,8.18,5
max,40.91892,-73.70623,0.869,908.3499999999999,519.6,11998.0,1752.6,89.63,49


summary,latitude,longitude,hour,severity_score,victim_count,estimated_police_eta_min,public_visibility_score
min,40.4963,-74.25396,0,1.0,0,1.5,0.01
25%,40.63115,-73.9731,4,3.29,0,5.2,0.307
50%,40.73483,-73.93187,14,5.25,0,6.9,0.465
75%,40.82293,-73.86586,20,7.17,1,9.0,0.625
max,40.91423,-73.71052,23,10.0,13,45.0,0.99


summary,temperature_c,precipitation_mm,wind_kmh,visibility_m,cloud_cover
min,-7.5,0.0,0.1,154.0,0.0
25%,6.3,0.0,6.2,6627.0,0.21
50%,12.8,0.0,11.2,8417.0,0.36
75%,18.7,0.0,18.8,10201.0,0.53
max,35.9,31.33,69.1,11998.0,1.0


In [0]:
%sql
SELECT
  -- crime próximo vs. campos de crime
  SUM(CASE WHEN crime_nearby = false AND nearest_crime_id IS NOT NULL THEN 1 ELSE 0 END) AS sem_crime_mas_com_id,
  SUM(CASE WHEN crime_nearby = true  AND nearest_crime_id IS NULL     THEN 1 ELSE 0 END) AS com_crime_mas_sem_id,
  -- foto vs. qualidade da foto
  SUM(CASE WHEN photo_evidence = false AND mean_photo_quality IS NOT NULL THEN 1 ELSE 0 END) AS sem_foto_mas_com_qualidade,
  SUM(CASE WHEN photo_evidence = true  AND mean_photo_quality IS NULL     THEN 1 ELSE 0 END) AS com_foto_mas_sem_qualidade,
  -- velocidade e altitude vs. número de testemunhas
  ROUND(AVG(CASE WHEN estimated_speed_kmh IS NULL THEN witness_count END), 2) AS testemunhas_medias_sem_velocidade,
  ROUND(AVG(CASE WHEN estimated_speed_kmh IS NOT NULL THEN witness_count END), 2) AS testemunhas_medias_com_velocidade,
  ROUND(AVG(CASE WHEN estimated_altitude_m IS NULL THEN witness_count END), 2) AS testemunhas_medias_sem_altitude,
  ROUND(AVG(CASE WHEN estimated_altitude_m IS NOT NULL THEN witness_count END), 2) AS testemunhas_medias_com_altitude
FROM workspace.bronze.sightings;

sem_crime_mas_com_id,com_crime_mas_sem_id,sem_foto_mas_com_qualidade,com_foto_mas_sem_qualidade,testemunhas_medias_sem_velocidade,testemunhas_medias_com_velocidade,testemunhas_medias_sem_altitude,testemunhas_medias_com_altitude
0,0,0,0,6.15,3.68,4.87,3.46


In [0]:
%sql
SELECT 'sem velocidade' AS grupo, COUNT(*) AS qtd, ROUND(AVG(witness_count), 2) AS testemunhas_medias
FROM workspace.bronze.sightings WHERE estimated_speed_kmh IS NULL
UNION ALL
SELECT 'com velocidade', COUNT(*), ROUND(AVG(witness_count), 2)
FROM workspace.bronze.sightings WHERE estimated_speed_kmh IS NOT NULL
UNION ALL
SELECT 'sem altitude', COUNT(*), ROUND(AVG(witness_count), 2)
FROM workspace.bronze.sightings WHERE estimated_altitude_m IS NULL
UNION ALL
SELECT 'com altitude', COUNT(*), ROUND(AVG(witness_count), 2)
FROM workspace.bronze.sightings WHERE estimated_altitude_m IS NOT NULL;

grupo,qtd,testemunhas_medias
sem velocidade,15152,6.15
com velocidade,71479,3.68
sem altitude,40074,4.87
com altitude,46557,3.46


In [0]:
%sql
SELECT
  report_type,
  COUNT(*) AS qtd,
  ROUND(AVG(CASE WHEN estimated_speed_kmh IS NULL THEN 1 ELSE 0 END) * 100, 1) AS pct_sem_velocidade,
  ROUND(AVG(CASE WHEN estimated_altitude_m IS NULL THEN 1 ELSE 0 END) * 100, 1) AS pct_sem_altitude,
  ROUND(AVG(CASE WHEN video_evidence THEN 1 ELSE 0 END) * 100, 1) AS pct_com_video
FROM workspace.bronze.sightings
GROUP BY report_type
ORDER BY qtd DESC;

report_type,qtd,pct_sem_velocidade,pct_sem_altitude,pct_com_video
street,24922,0.0,100.0,34.8
swinging,16519,0.0,0.0,19.3
stationary,15152,100.0,100.0,44.4
rooftop,14062,0.0,0.0,26.8
rescue,6689,0.0,0.0,55.5
pursuit,5013,0.0,0.0,22.8
crime_response,4274,0.0,0.0,26.4


SILVER

In [0]:
from pyspark.sql import functions as F

def limpar_textos(df):
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(c, F.trim(F.col(f"`{c}`")))
    return df

# --- avistamentos ---
s = limpar_textos(spark.table("workspace.bronze.sightings")).dropDuplicates(["sighting_id"])
s = (s.withColumn("data", F.to_date("timestamp"))
      .withColumn("hora", F.hour("timestamp"))
      .withColumn("periodo_do_dia",
          F.when(F.col("hora") < 6, "madrugada")
           .when(F.col("hora") < 12, "manha")
           .when(F.col("hora") < 18, "tarde")
           .otherwise("noite"))
      .withColumn("tem_velocidade", F.col("estimated_speed_kmh").isNotNull())
      .withColumn("tem_altitude", F.col("estimated_altitude_m").isNotNull()))
s.write.mode("overwrite").saveAsTable("workspace.silver.avistamentos")

# --- crimes ---
c = limpar_textos(spark.table("workspace.bronze.crime_incidents")).dropDuplicates(["crime_id"])
c.write.mode("overwrite").saveAsTable("workspace.silver.crimes")

# --- locais ---
l = limpar_textos(spark.table("workspace.bronze.locations")).dropDuplicates(["zone_id"])
l.write.mode("overwrite").saveAsTable("workspace.silver.locais")

# --- clima ---
w = limpar_textos(spark.table("workspace.bronze.weather")).dropDuplicates(["weather_id"])
w.write.mode("overwrite").saveAsTable("workspace.silver.clima")

# --- testemunhas ---
t = limpar_textos(spark.table("workspace.bronze.witnesses_public")).dropDuplicates(["witness_id"])
t.write.mode("overwrite").saveAsTable("workspace.silver.testemunhas")

for nome in ["avistamentos", "crimes", "locais", "clima", "testemunhas"]:
    print(nome, spark.table(f"workspace.silver.{nome}").count())

avistamentos 86631
crimes 46000
locais 46
clima 13104
testemunhas 27000


In [0]:
import pandas as pd
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)

def q(sql):
    return spark.sql(sql).toPandas().to_string(index=False)

print("=== 1. JOINs de avistamentos ===")
print(q("""
SELECT COUNT(*) AS total,
  SUM(CASE WHEN l.h3_cell IS NULL THEN 1 ELSE 0 END) AS sem_local,
  SUM(CASE WHEN s.nearest_crime_id IS NOT NULL THEN 1 ELSE 0 END) AS com_crime,
  SUM(CASE WHEN s.nearest_crime_id IS NOT NULL AND c.crime_id IS NULL THEN 1 ELSE 0 END) AS crime_nao_encontrado,
  SUM(CASE WHEN w.weather_id IS NULL THEN 1 ELSE 0 END) AS sem_clima
FROM workspace.silver.avistamentos s
LEFT JOIN workspace.silver.locais l ON s.h3_cell = l.h3_cell
LEFT JOIN workspace.silver.crimes c ON s.nearest_crime_id = c.crime_id
LEFT JOIN workspace.silver.clima w ON s.data = w.date AND s.hora = w.hour
"""))

print("\n=== 2. Avistamentos e crimes por borough ===")
print(q("SELECT borough, COUNT(*) AS avistamentos FROM workspace.silver.avistamentos GROUP BY borough ORDER BY 2 DESC"))
print(q("SELECT borough, COUNT(*) AS crimes FROM workspace.silver.crimes GROUP BY borough ORDER BY 2 DESC"))

print("\n=== 3. Valores categóricos ===")
for tab, col in [("avistamentos","report_type"), ("avistamentos","verification_status"),
                 ("avistamentos","report_source"), ("avistamentos","weather_condition"),
                 ("avistamentos","movement_direction"), ("crimes","crime_type"),
                 ("crimes","time_of_day"), ("clima","condition"),
                 ("testemunhas","preferred_report_channel"), ("testemunhas","home_borough")]:
    print(f"--- {tab}.{col} ---")
    print(q(f"SELECT {col}, COUNT(*) AS qtd FROM workspace.silver.{tab} GROUP BY {col} ORDER BY 2 DESC"))

print("\n=== 4. Mínimos e máximos ===")
for tab, cols in [
    ("avistamentos", ["latitude","longitude","tracker_confidence","estimated_speed_kmh","estimated_altitude_m",
                      "visibility_m","distance_to_nearest_crime_m","minutes_from_nearest_crime","witness_count","crowd_density_score"]),
    ("crimes", ["latitude","longitude","hour","severity_score","victim_count","estimated_police_eta_min","public_visibility_score"]),
    ("clima", ["temperature_c","precipitation_mm","wind_kmh","visibility_m","cloud_cover","hour"]),
    ("testemunhas", ["total_reports","reports_with_photo","avg_statement_certainty"]),
    ("locais", ["population_density_score","nightlife_score","commercial_score","residential_score","patrol_activity_score"]),
]:
    print(f"--- {tab} ---")
    exprs = ", ".join([f"MIN({c}) AS min_{c}, MAX({c}) AS max_{c}" for c in cols])
    r = spark.sql(f"SELECT {exprs} FROM workspace.silver.{tab}").first().asDict()
    for c in cols:
        print(f"{c}: min={r['min_'+c]} | max={r['max_'+c]}")

print("\n=== 5. Nulos (witnesses_public, locations, weather) ===")
for nome in ["witnesses_public", "locations", "weather"]:
    t = spark.table(f"workspace.bronze.{nome}")
    total = t.count()
    cols = [c for c in t.columns if not c.startswith("_")]
    cont = t.select([F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in cols]).first().asDict()
    nul = {c: f"{v/total*100:.2f}%" for c, v in cont.items() if v}
    print(f"{nome}: {nul if nul else 'nenhuma coluna com nulos'}")

print("\n=== 6. Consistência ===")
print(q("""
SELECT
  SUM(CASE WHEN is_verified = 1 AND verification_status NOT LIKE '%verif%' THEN 1 ELSE 0 END) AS verificado_status_estranho,
  SUM(CASE WHEN witness_count < unique_source_count THEN 1 ELSE 0 END) AS fontes_maior_que_testemunhas
FROM workspace.silver.avistamentos
"""))

=== 1. JOINs de avistamentos ===
 total  sem_local  com_crime  crime_nao_encontrado  sem_clima
 86631      78177       6072                     0          0

=== 2. Avistamentos e crimes por borough ===
      borough  avistamentos
    Manhattan         24009
     Brooklyn         23157
       Queens         19158
        Bronx         13957
Staten_Island          6350
      borough  crimes
    Manhattan   13713
     Brooklyn    9786
       Queens    9410
        Bronx    7712
Staten_Island    5379

=== 3. Valores categóricos ===
--- avistamentos.report_type ---
   report_type   qtd
        street 24922
      swinging 16519
    stationary 15152
       rooftop 14062
        rescue  6689
       pursuit  5013
crime_response  4274
--- avistamentos.verification_status ---
verification_status   qtd
           verified 54611
  mistaken_identity 10743
       impersonator  8066
   duplicate_report  4802
  social_media_hoax  4491
    deliberate_fake  3291
       sensor_error   627
--- avistamento

In [0]:
print("=== A. h3_cell: avistamentos x locais ===")
print(q("""SELECT COUNT(DISTINCT h3_cell) AS h3_distintos_avistamentos,
                 (SELECT COUNT(DISTINCT h3_cell) FROM workspace.silver.locais) AS h3_distintos_locais
          FROM workspace.silver.avistamentos"""))
print(q("SELECT zone_id, borough, district, h3_cell FROM workspace.silver.locais ORDER BY borough, district LIMIT 12"))
print(q("SELECT h3_cell, borough, district FROM workspace.silver.avistamentos LIMIT 5"))

print("\n=== B. JOIN por borough + district ===")
print(q("""SELECT COUNT(*) AS total,
                 SUM(CASE WHEN l.zone_id IS NULL THEN 1 ELSE 0 END) AS sem_local
          FROM workspace.silver.avistamentos s
          LEFT JOIN workspace.silver.locais l ON s.borough = l.borough AND s.district = l.district"""))
print(q("""SELECT COUNT(*) AS linhas_locais,
                 COUNT(DISTINCT borough, district) AS combinacoes_borough_district
          FROM workspace.silver.locais"""))

print("\n=== C. Mapeamento de time_of_day em crimes ===")
print(q("""SELECT time_of_day, MIN(hour) AS hora_min, MAX(hour) AS hora_max, COUNT(*) AS qtd
          FROM workspace.silver.crimes GROUP BY time_of_day ORDER BY hora_min"""))

print("\n=== D. Velocidade ===")
print(q("""SELECT percentile_approx(estimated_speed_kmh, array(0.5, 0.9, 0.99, 0.999)) AS percentis,
                 SUM(CASE WHEN estimated_speed_kmh > 200 THEN 1 ELSE 0 END) AS acima_200,
                 SUM(CASE WHEN estimated_speed_kmh > 500 THEN 1 ELSE 0 END) AS acima_500
          FROM workspace.silver.avistamentos"""))
print(q("""SELECT report_type, ROUND(AVG(estimated_speed_kmh),1) AS media, MAX(estimated_speed_kmh) AS maximo
          FROM workspace.silver.avistamentos GROUP BY report_type ORDER BY maximo DESC"""))

print("\n=== E. is_verified x verification_status ===")
print(q("""SELECT verification_status, is_verified, COUNT(*) AS qtd
          FROM workspace.silver.avistamentos GROUP BY 1, 2 ORDER BY 1, 2"""))

print("\n=== F. Testemunhas com nulo ou canal 'none' ===")
print(q("""SELECT * FROM workspace.silver.testemunhas
          WHERE avg_statement_certainty IS NULL OR preferred_report_channel = 'none'"""))

=== A. h3_cell: avistamentos x locais ===
 h3_distintos_avistamentos  h3_distintos_locais
                      1513                   46
   zone_id  borough           district         h3_cell
ZON_000033    Bronx    Bronx_Central_0 882a100a97fffff
ZON_000036    Bronx       Bronx_East_3 882a100acbfffff
ZON_000034    Bronx      Bronx_North_1 882a1001abfffff
ZON_000035    Bronx  Bronx_Northeast_2 882a1001d5fffff
ZON_000041    Bronx  Bronx_Northwest_8 882a100337fffff
ZON_000038    Bronx      Bronx_South_5 882a100021fffff
ZON_000037    Bronx  Bronx_Southeast_4 882a10013bfffff
ZON_000039    Bronx  Bronx_Southwest_6 882a10012dfffff
ZON_000040    Bronx       Bronx_West_7 882a100147fffff
ZON_000009 Brooklyn Brooklyn_Central_0 882a107427fffff
ZON_000018 Brooklyn Brooklyn_Central_9 882a107749fffff
ZON_000012 Brooklyn    Brooklyn_East_3 882a1076cdfffff
        h3_cell  borough             district
882a100e05fffff   Queens      Queens_North_10
882a107733fffff Brooklyn Brooklyn_Southeast_4
882a1001a

In [0]:
from pyspark.sql import functions as F

def limpar_textos(df):
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(c, F.trim(F.col(f"`{c}`")))
    return df

def periodo(hora):  # mesmo mapeamento observado em crime_incidents.time_of_day
    return (F.when((hora >= 5) & (hora <= 11), "morning")
             .when((hora >= 12) & (hora <= 16), "afternoon")
             .when((hora >= 17) & (hora <= 21), "evening")
             .otherwise("night"))

# --- avistamentos ---
s = limpar_textos(spark.table("workspace.bronze.sightings")).dropDuplicates(["sighting_id"])
s = (s.withColumn("data", F.to_date("timestamp"))
      .withColumn("hora", F.hour("timestamp"))
      .withColumn("time_of_day", periodo(F.col("hora")))
      .withColumn("tem_velocidade", F.col("estimated_speed_kmh").isNotNull())
      .withColumn("tem_altitude", F.col("estimated_altitude_m").isNotNull())
      .withColumn("velocidade_suspeita", F.coalesce(F.col("estimated_speed_kmh") > 250, F.lit(False))))
(s.write.mode("overwrite").option("overwriteSchema", "true")
  .saveAsTable("workspace.silver.avistamentos"))

# --- testemunhas ---
t = limpar_textos(spark.table("workspace.bronze.witnesses_public")).dropDuplicates(["witness_id"])
t = t.withColumn("sem_relatos", F.col("total_reports") == 0)
(t.write.mode("overwrite").option("overwriteSchema", "true")
  .saveAsTable("workspace.silver.testemunhas"))

print("avistamentos:", spark.table("workspace.silver.avistamentos").count())
print("testemunhas:", spark.table("workspace.silver.testemunhas").count())

avistamentos: 86631
testemunhas: 27000


GOLD

In [0]:
from pyspark.sql import functions as F
import pandas as pd
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)

def q(sql):
    return spark.sql(sql).toPandas().to_string(index=False)

loc = spark.table("workspace.silver.locais").select("zone_id", "borough", "district")
clima = spark.table("workspace.silver.clima").select(
    "weather_id", F.col("date").alias("data"), F.col("hour").alias("hora"))

# --- dim_local ---
(spark.table("workspace.silver.locais")
   .withColumnRenamed("h3_cell", "h3_cell_zona")
   .select("zone_id", "borough", "district", "h3_cell_zona", "centroid_lat", "centroid_lon",
           "population_density_score", "nightlife_score", "commercial_score",
           "residential_score", "patrol_activity_score")
   .write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.gold.dim_local"))

# --- dim_clima ---
(spark.table("workspace.silver.clima")
   .withColumnRenamed("date", "data").withColumnRenamed("hour", "hora")
   .write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.gold.dim_clima"))

# --- fato_avistamentos ---
fa = (spark.table("workspace.silver.avistamentos")
        .join(loc, ["borough", "district"], "left")
        .join(clima, ["data", "hora"], "left")
        .select("sighting_id", "zone_id", "weather_id", "nearest_crime_id", "timestamp", "data", "hora",
                "time_of_day", "report_type", "report_source", "verification_status", "is_verified",
                "witness_count", "unique_source_count", "tracker_confidence", "photo_evidence",
                "video_evidence", "crime_nearby", "distance_to_nearest_crime_m",
                "minutes_from_nearest_crime", "estimated_speed_kmh", "velocidade_suspeita",
                "tem_velocidade", "estimated_altitude_m", "tem_altitude", "movement_direction",
                "latitude", "longitude", "h3_cell"))
(fa.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.gold.fato_avistamentos"))

# --- fato_crimes ---
fc = (spark.table("workspace.silver.crimes")
        .withColumnRenamed("date", "data").withColumnRenamed("hour", "hora")
        .join(loc, ["borough", "district"], "left")
        .join(clima, ["data", "hora"], "left")
        .select("crime_id", "zone_id", "weather_id", "timestamp", "data", "hora", "time_of_day",
                "crime_type", "severity_score", "victim_count", "weapon_reported", "police_called",
                "estimated_police_eta_min", "public_visibility_score", "is_decoy_event",
                "latitude", "longitude", "h3_cell"))
(fc.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.gold.fato_crimes"))

# --- verificação ---
print("=== Contagens ===")
for t in ["dim_local", "dim_clima", "fato_avistamentos", "fato_crimes"]:
    print(t, spark.table(f"workspace.gold.{t}").count())

print("\n=== Chaves nulas ===")
for t in ["fato_avistamentos", "fato_crimes"]:
    print(t)
    print(q(f"""SELECT COUNT(*) AS total,
                       SUM(CASE WHEN zone_id IS NULL THEN 1 ELSE 0 END) AS sem_zona,
                       SUM(CASE WHEN weather_id IS NULL THEN 1 ELSE 0 END) AS sem_clima
                FROM workspace.gold.{t}"""))

print("\n=== Velocidade suspeita por status ===")
print(q("""SELECT verification_status, COUNT(*) AS suspeitos
           FROM workspace.gold.fato_avistamentos WHERE velocidade_suspeita
           GROUP BY 1 ORDER BY 2 DESC"""))

=== Contagens ===
dim_local 46
dim_clima 13104
fato_avistamentos 86631
fato_crimes 46000

=== Chaves nulas ===
fato_avistamentos
 total  sem_zona  sem_clima
 86631         0          0
fato_crimes
 total  sem_zona  sem_clima
 46000         0          0

=== Velocidade suspeita por status ===
verification_status  suspeitos
           verified        129
   duplicate_report          6


In [0]:
def esc(s):
    return s.replace("'", "''")

catalogo = {
 "fato_avistamentos": ("Um registro por avistamento do Homem-Aranha (86.631). Origem: sightings.csv do Kaggle (bronze.sightings -> silver.avistamentos), enriquecido com zona (JOIN por borough+district) e clima (JOIN por data+hora).", {
  "sighting_id": "Identificador unico do avistamento (chave primaria).",
  "zone_id": "Zona do avistamento, chave para dim_local. Vem do JOIN por borough e district.",
  "weather_id": "Registro de clima da hora do avistamento, chave para dim_clima. Vem do JOIN por data e hora.",
  "nearest_crime_id": "Crime mais proximo, chave para fato_crimes. Nulo (cerca de 93%) quando nao ha crime proximo.",
  "timestamp": "Data e hora do avistamento.",
  "data": "Data do avistamento. Derivada do timestamp na silver.",
  "hora": "Hora do dia (0 a 23). Derivada do timestamp na silver.",
  "time_of_day": "Periodo do dia derivado da hora: morning (5h-11h), afternoon (12h-16h), evening (17h-21h) ou night (22h-4h). Mesmo criterio de fato_crimes.",
  "report_type": "Tipo de relato: street, swinging, stationary, rooftop, rescue, pursuit ou crime_response.",
  "report_source": "Fonte do relato: mobile_app, social_media, police_scanner, anonymous_tip, camera_network ou news_report.",
  "verification_status": "Status de verificacao: verified, mistaken_identity, impersonator, duplicate_report, social_media_hoax, deliberate_fake ou sensor_error.",
  "is_verified": "1 se o avistamento e verdadeiro (status verified), 0 caso contrario.",
  "witness_count": "Numero de testemunhas do avistamento (1 a 49).",
  "unique_source_count": "Numero de fontes distintas do relato (nunca maior que witness_count).",
  "tracker_confidence": "Confianca do rastreador no avistamento (0,21 a 0,87). Significado inferido do nome da coluna.",
  "photo_evidence": "True se ha foto como evidencia.",
  "video_evidence": "True se ha video como evidencia.",
  "crime_nearby": "True se ha um crime associado ao avistamento (cerca de 7% dos casos).",
  "distance_to_nearest_crime_m": "Distancia em metros ate o crime mais proximo (0 a 1.753). Nulo quando crime_nearby e false.",
  "minutes_from_nearest_crime": "Minutos entre o avistamento e o crime (-89 a +89). Negativo = avistado antes do crime. Nulo sem crime proximo.",
  "estimated_speed_kmh": "Velocidade estimada em km/h (0 a 908). Nulo em relatos stationary.",
  "velocidade_suspeita": "True se a velocidade estimada passa de 250 km/h (valor extremo, mantido no dado). Coluna criada na silver.",
  "tem_velocidade": "True se estimated_speed_kmh nao e nulo. Coluna criada na silver.",
  "estimated_altitude_m": "Altitude estimada em metros (0 a 520). Nulo em relatos stationary e street.",
  "tem_altitude": "True se estimated_altitude_m nao e nulo. Coluna criada na silver.",
  "movement_direction": "Direcao do movimento: N, NE, E, SE, S, SW, W ou NW.",
  "latitude": "Latitude do avistamento (40,49 a 40,92, dentro de Nova York).",
  "longitude": "Longitude do avistamento (-74,26 a -73,71, dentro de Nova York).",
  "h3_cell": "Codigo H3 (hexagono geografico) do local do avistamento.",
 }),
 "fato_crimes": ("Um registro por crime (46.000). Origem: crime_incidents.csv do Kaggle (bronze.crime_incidents -> silver.crimes), enriquecido com zona e clima.", {
  "crime_id": "Identificador unico do crime (chave primaria).",
  "zone_id": "Zona do crime, chave para dim_local (JOIN por borough e district).",
  "weather_id": "Registro de clima da hora do crime, chave para dim_clima (JOIN por data e hora).",
  "timestamp": "Data e hora do crime.",
  "data": "Data do crime (coluna date da origem, renomeada).",
  "hora": "Hora do crime, 0 a 23 (coluna hour da origem, renomeada).",
  "time_of_day": "Periodo do dia: morning (5h-11h), afternoon (12h-16h), evening (17h-21h) ou night (22h-4h).",
  "crime_type": "Tipo: vehicle_theft, burglary, assault, major_incident, pursuit, suspicious_activity, property_damage, robbery ou armed_robbery.",
  "severity_score": "Gravidade do crime, de 1 a 10.",
  "victim_count": "Numero de vitimas (0 a 13).",
  "weapon_reported": "True se houve relato de arma.",
  "police_called": "True se a policia foi acionada.",
  "estimated_police_eta_min": "Tempo estimado de chegada da policia em minutos (1,5 a 45).",
  "public_visibility_score": "Pontuacao de visibilidade publica do crime (0,01 a 0,99). Significado inferido do nome da coluna.",
  "is_decoy_event": "Indica se o evento e uma isca (decoy). Significado inferido do nome da coluna.",
  "latitude": "Latitude do crime (40,50 a 40,91).",
  "longitude": "Longitude do crime (-74,25 a -73,71).",
  "h3_cell": "Codigo H3 (hexagono geografico) do local do crime.",
 }),
 "dim_local": ("Dimensao de zonas de Nova York (46 linhas, uma por distrito). Origem: locations.csv do Kaggle (bronze.locations -> silver.locais).", {
  "zone_id": "Identificador unico da zona (chave primaria).",
  "borough": "Borough: Manhattan, Brooklyn, Queens, Bronx ou Staten_Island.",
  "district": "Distrito dentro do borough (por exemplo Bronx_North_1). Unico por zona junto com borough.",
  "h3_cell_zona": "Codigo H3 do centro da zona (h3_cell na origem, renomeado).",
  "centroid_lat": "Latitude do centro da zona.",
  "centroid_lon": "Longitude do centro da zona.",
  "population_density_score": "Densidade populacional da zona, de 0,07 a 1,0.",
  "nightlife_score": "Intensidade de vida noturna da zona, de 0,02 a 0,90.",
  "commercial_score": "Perfil comercial da zona, de 0,04 a 1,0.",
  "residential_score": "Perfil residencial da zona, de 0,52 a 1,0.",
  "patrol_activity_score": "Intensidade de patrulha policial da zona, de 0,07 a 0,66.",
 }),
 "dim_clima": ("Dimensao de clima por hora (13.104 linhas). Origem: weather.csv do Kaggle (bronze.weather -> silver.clima).", {
  "weather_id": "Identificador unico do registro de clima (chave primaria).",
  "data": "Data do registro (coluna date da origem, renomeada).",
  "hora": "Hora do registro, 0 a 23 (coluna hour da origem, renomeada).",
  "temperature_c": "Temperatura em graus Celsius (-7,5 a 35,9).",
  "precipitation_mm": "Precipitacao em mm (0 a 31,3).",
  "wind_kmh": "Velocidade do vento em km/h (0,1 a 69,1).",
  "visibility_m": "Visibilidade em metros (154 a 11.998).",
  "cloud_cover": "Cobertura de nuvens, de 0 a 1.",
  "condition": "Condicao: clear, cloudy, rain, snow, fog ou storm.",
 }),
}

for tabela, (desc_tabela, colunas) in catalogo.items():
    nome = f"workspace.gold.{tabela}"
    tipos = dict(spark.table(nome).dtypes)
    spark.sql(f"COMMENT ON TABLE {nome} IS '{esc(desc_tabela)}'")
    for col, desc in colunas.items():
        texto = f"{desc} [tipo: {tipos[col]}]"
        spark.sql(f"ALTER TABLE {nome} ALTER COLUMN `{col}` COMMENT '{esc(texto)}'")
    faltando = set(tipos) - set(colunas)
    print(tabela, "ok", "| colunas sem descricao:", faltando or "nenhuma")

fato_avistamentos ok | colunas sem descricao: nenhuma
fato_crimes ok | colunas sem descricao: nenhuma
dim_local ok | colunas sem descricao: nenhuma
dim_clima ok | colunas sem descricao: {'_ingestao_em', '_arquivo_origem'}


In [0]:
nome = "workspace.gold.dim_clima"
tipos = dict(spark.table(nome).dtypes)

descricoes = {
    "_ingestao_em": "Data e hora em que o registro foi carregado na camada bronze (coluna de controle).",
    "_arquivo_origem": "Arquivo de origem do registro (weather.csv). Coluna de controle criada na bronze.",
}
for col, desc in descricoes.items():
    texto = f"{desc} [tipo: {tipos[col]}]"
    spark.sql(f"ALTER TABLE {nome} ALTER COLUMN `{col}` COMMENT '{texto}'")

print("ok")

ok
